# 04 — Le maschere: su che cosa si reggono i due modelli (tentativo 13, fase 2)

Sugli articoli delle testate valutate (al più 50 per testata, seed fisso), con i modelli dei fold in cui erano nel test:
si **tolgono** dal testo le parole di una categoria (come in `01`) e si misura quanto peggiorano i due modelli, per
testata (feature: voto degli articoli; ModernBERT: media delle probabilità), come nella fase 1. Condizioni:

- **originale**;
- **senza nomi propri e termini di parte**;
- **senza argomento** (i sostantivi che non sono in altre categorie);
- **senza cornice del sito**;
- **controllo a caso**: tolte tante parole quante "nomi e termini di parte", ma scelte a caso tra le altre (per
  separare l'effetto di *quali* parole si tolgono da quello di *quante*).

Metriche: F1 macro, ±1 livello, binario; la **variazione** rispetto all'originale (mascherato − originale: negativa se il
modello peggiora), con intervalli al 95%, e la differenza tra le variazioni dei due modelli (ModernBERT − feature), con un
bootstrap per gruppo; per tutti e per orientamento. Uscite in `Results/3-xai/maschere/`.

In [ ]:
PROVA = False      # True: fold 1, 1 articolo per testata

In [ ]:
import time

import joblib
import numpy as np
import pandas as pd
import spacy
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import sys; sys.path.insert(0, "../2-modelli")
from comune import metriche_di, per_testata
from comune_xai import LIVELLI, MAX_TOKEN, MODELLI, RESULTS, SEED, VERSIONI, categoria, leggi_lessici, termini_di_parte, togli

In [ ]:
OUT = RESULTS / ("prova" if PROVA else "") / "maschere"; OUT.mkdir(parents=True, exist_ok=True)
CLASSI = [f"p_l{k}" for k in range(1, 6)]
art = pd.read_csv(VERSIONI / "dataset.csv"); art["text"] = art.text.fillna("")
testate = pd.read_csv(VERSIONI / "fold.csv").set_index("domain")
val = testate[testate.valutazione]
if PROVA:
    val = val[val.fold_gruppo == 1]
per_testata_max = 1 if PROVA else 50
a = art[art.domain.isin(val.index)].sample(frac=1, random_state=SEED)
a = a[a.groupby("domain").cumcount() < per_testata_max]             # al più per_testata_max articoli a caso per testata
a = a.assign(fold=a.domain.map(testate.fold_gruppo)).reset_index(drop=True)
lessici = leggi_lessici()
DI_PARTE = {f: termini_di_parte(art, testate, f)[0] for f in sorted(a.fold.unique())}
nlp = spacy.load("en_core_web_sm")
rng = np.random.default_rng(SEED)
CONDIZIONI = ["originale", "senza nomi e termini di parte", "senza argomento", "senza cornice del sito", "controllo a caso"]
testi = {c: [] for c in CONDIZIONI}
t0 = time.time()
for (f, testo), doc in zip(a[["fold", "text"]].itertuples(index=False), nlp.pipe(a.text.tolist(), batch_size=32, n_process=16)):
    cat = [(t.idx, t.idx + len(t.text), categoria(t, lessici, DI_PARTE[f])) for t in doc if not t.is_space]
    span = lambda nomi: [(s, e) for s, e, c in cat if c in nomi]
    nomi = span({"nomi propri", "termini di parte"})
    altri = [(s, e) for s, e, c in cat if c not in {"nomi propri", "termini di parte"}]
    caso = [altri[i] for i in rng.choice(len(altri), min(len(nomi), len(altri)), replace=False)] if altri else []
    testi["originale"].append(testo)
    testi["senza nomi e termini di parte"].append(togli(testo, nomi))
    testi["senza argomento"].append(togli(testo, span({"argomento"})))
    testi["senza cornice del sito"].append(togli(testo, span({"cornice del sito"})))
    testi["controllo a caso"].append(togli(testo, caso))
print(f"{len(a)} articoli di {a.domain.nunique()} testate, testi mascherati in {(time.time() - t0) / 60:.1f} min")
lung = pd.DataFrame({c: [len(x.split()) for x in v] for c, v in testi.items()})
print("parole per articolo (mediana):", lung.median().to_dict())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
while torch.cuda.is_available() and (torch.cuda.mem_get_info()[0] + torch.cuda.memory_reserved()) / 2**30 < 6.0:
    print("GPU occupata da altri: riprovo tra 5 minuti", flush=True); time.sleep(300)
P = {(m, c): np.zeros((len(a), 5)) for m in ["feature", "ModernBERT"] for c in CONDIZIONI}
t0 = time.time()
for f in sorted(a.fold.unique()):
    ix = np.where(a.fold == f)[0]
    mf = joblib.load(MODELLI / "feature" / f"fold{f}.joblib")
    d = MODELLI / "modernbert" / f"fold{f}"
    tk = AutoTokenizer.from_pretrained(d)
    mb = AutoModelForSequenceClassification.from_pretrained(d, reference_compile=False).to(device).eval()
    for c in CONDIZIONI:
        tt = [testi[c][i] for i in ix]
        p = np.zeros((len(ix), 5)); p[:, mf["modello"].classes_] = mf["modello"].predict_proba(mf["tfidf"].transform(tt))
        P[("feature", c)][ix] = p
        ordine = np.argsort([len(x) for x in tt]); out = np.zeros((len(ix), 5))
        for s in range(0, len(ix), 16):
            pezzo = ordine[s:s + 16]
            enc = tk([tt[i] for i in pezzo], truncation=True, max_length=MAX_TOKEN, padding=True, return_tensors="pt").to(device)
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                out[pezzo] = torch.softmax(mb(**enc).logits.float(), -1).cpu().numpy()
        P[("ModernBERT", c)][ix] = out
    del mb; torch.cuda.empty_cache()
    print(f"fold {f}: fatto ({(time.time() - t0) / 60:.1f} min)", flush=True)
righe = a[["article_id", "domain", "fold", "label"]].copy()
for (m, c), p in P.items():
    for k in range(5):
        righe[f"{m} | {c} | p_l{k + 1}"] = p[:, k]
righe.to_csv(OUT / "probabilita_articoli.csv.gz", index=False)

In [ ]:
def per_testata_modello(m, c):
    p = pd.DataFrame(P[(m, c)], columns=CLASSI).assign(domain=a.domain.values, y_true=a.label.values, y_pred=P[(m, c)].argmax(1))
    if m == "feature":
        return per_testata(p).y_pred                             # voto degli articoli, come nella fase 1
    return p.groupby("domain")[CLASSI].mean().pipe(lambda x: pd.Series(x.to_numpy().argmax(1), index=x.index))


def quattro(y, p):
    M = np.bincount(y * 5 + p, minlength=25).reshape(5, 5)
    tp = np.diag(M); den = M.sum(0) + M.sum(1)
    d = np.abs(y - p)
    return [np.divide(2 * tp, den, out=np.zeros(5), where=den > 0).mean(), (d <= 1).mean(), ((y >= 2) == (p >= 2)).mean()]


METR = ["f1_macro", "tolerant_1", "bin_accuratezza"]
doms = sorted(a.domain.unique()); v = val.loc[doms]
PRED = {(m, c): per_testata_modello(m, c).reindex(doms).to_numpy().astype(int) for m in ["feature", "ModernBERT"] for c in CONDIZIONI}
righe, diff = [], []
for gruppo in ["tutte", "Right", "Left", "nessuno"]:
    sel = np.ones(len(v), bool) if gruppo == "tutte" else (v.orientamento == gruppo).to_numpy()
    y = v.label.to_numpy()[sel]; g = v.gruppo.to_numpy()[sel]
    gr = np.unique(g); pos = {x: np.where(g == x)[0] for x in gr}; rng = np.random.default_rng(0)
    camp = [np.concatenate([pos[x] for x in rng.choice(gr, len(gr))]) for _ in range(1000)]
    BS = {k: np.array([quattro(y[i], p[sel][i]) for i in camp]) for k, p in PRED.items()}
    for (m, c), p in PRED.items():
        mm = quattro(y, p[sel]); cal = np.array(mm) - np.array(quattro(y, PRED[(m, "originale")][sel])); bc = BS[(m, c)] - BS[(m, "originale")]
        righe.append({"testate": gruppo, "n": int(sel.sum()), "modello": m, "condizione": c, **dict(zip(METR, mm)),
                      **{f"variazione {k}": cal[j] for j, k in enumerate(METR)}, **{f"variazione {k} 2.5%": np.percentile(bc[:, j], 2.5) for j, k in enumerate(METR)},
                      **{f"variazione {k} 97.5%": np.percentile(bc[:, j], 97.5) for j, k in enumerate(METR)}})
    for c in CONDIZIONI[1:]:
        dd = (BS[("ModernBERT", c)] - BS[("ModernBERT", "originale")]) - (BS[("feature", c)] - BS[("feature", "originale")])
        cm = (np.array(quattro(y, PRED[("ModernBERT", c)][sel])) - quattro(y, PRED[("ModernBERT", "originale")][sel])) - \
             (np.array(quattro(y, PRED[("feature", c)][sel])) - quattro(y, PRED[("feature", "originale")][sel]))
        diff.append({"testate": gruppo, "condizione": c, **{f"Δ variazione {k} (ModernBERT − feature)": cm[j] for j, k in enumerate(METR)},
                     **{f"Δ variazione {k} 2.5%": np.percentile(dd[:, j], 2.5) for j, k in enumerate(METR)}, **{f"Δ variazione {k} 97.5%": np.percentile(dd[:, j], 97.5) for j, k in enumerate(METR)}})
ris = pd.DataFrame(righe); ris.to_csv(OUT / "maschere.csv", index=False)
pd.DataFrame(diff).to_csv(OUT / "maschere_differenze.csv", index=False)
(OUT / "FATTO").write_text(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))
ris[ris.testate == "tutte"].pivot(index="condizione", columns="modello", values=["f1_macro", "variazione f1_macro", "bin_accuratezza"]).reindex(CONDIZIONI).round(3)